# Week 3 Day 7 — Transformer Encoder Block
**Jul 19, 2026**

Closing out Week 3: generalize Day 6's single-head self-attention to **multi-head**, then wrap it with the residual connections, layer norm, and feedforward network that turn "an attention mechanism" into "a transformer encoder block." Both pieces get checked against PyTorch's own `nn.MultiheadAttention` / `nn.TransformerEncoderLayer` as oracles, same verification pattern as Day 6.

Scaffold as usual: TODO stubs, hints not formulas, self-check cells.

## Part 1: Setup

Given. Same shape of toy data as Day 6, plus `num_heads` and a feedforward hidden size.

In [20]:
import torch
import torch.nn as nn
import math

torch.manual_seed(0)
batch, seq_len, d_model, num_heads, d_ff = 2, 5, 8, 2, 16

x = torch.randn(batch, seq_len, d_model)
print(x.shape)

torch.Size([2, 5, 8])


## Part 2: Multi-head attention

TODO: build `MultiHeadAttention(d_model, num_heads)`. Same four `nn.Linear(d_model, d_model)` layers as you'd guess (`q_proj`, `k_proj`, `v_proj`, plus a new `out_proj` applied at the very end) but the attention computation itself needs to happen **per head**, independently.

The mechanism: `d_model` must be evenly divisible by `num_heads` (call the quotient `head_dim`). After projecting `x` to `Q`/`K`/`V` (still shape `(batch, seq_len, d_model)`), reshape each into `(batch, seq_len, num_heads, head_dim)` — splitting the last dimension into `num_heads` separate chunks — then transpose so `num_heads` sits where `batch` normally would: `(batch, num_heads, seq_len, head_dim)`. Now Day 6's exact scaled dot-product attention formula (`Q @ K^T / sqrt(head_dim)`, softmax, `@ V`) works unchanged — the extra `num_heads` dimension just rides along as if it were part of the batch, giving every head its own independent attention pattern. Finally, transpose back and reshape to `(batch, seq_len, d_model)`, then pass through `out_proj` — a detail it's easy to forget, but every real multi-head attention implementation ends with this output projection mixing the heads' results back together.

In [24]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        # TODO: self.num_heads, self.head_dim, self.q_proj, self.k_proj, self.v_proj, self.out_proj
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        

    def forward(self, x):
        # TODO
        batch, seq_len, d_model = x.shape
        q = self.q_proj(x).view(batch, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(batch, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(batch, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        attn_weights = torch.matmul(q, k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn_weights = torch.softmax(attn_weights, dim=-1)
        out = torch.matmul(attn_weights, v)

        out = out.transpose(1, 2).contiguous().view(batch, seq_len, d_model)
        out = self.out_proj(out)

        return out, attn_weights

mha = MultiHeadAttention(d_model, num_heads)
out, attn_weights = mha(x)
assert out.shape == (batch, seq_len, d_model), f"expected {(batch, seq_len, d_model)}, got {tuple(out.shape)}"
print("shape OK")

shape OK


## Part 3: Check against `nn.MultiheadAttention`

Given. PyTorch's built-in packs `Q`/`K`/`V` weights into one concatenated matrix (`in_proj_weight`), so matching it requires copying your three separate projections into that layout — a PyTorch-specific detail, not conceptually important, so it's given rather than an exercise.

In [27]:
mha.eval()
with torch.no_grad():
    my_out, _ = mha(x)

ref_mha = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
ref_mha.eval()
with torch.no_grad():
    ref_mha.in_proj_weight.copy_(torch.cat([mha.q_proj.weight, mha.k_proj.weight, mha.v_proj.weight], dim=0))
    ref_mha.in_proj_bias.copy_(torch.cat([mha.q_proj.bias, mha.k_proj.bias, mha.v_proj.bias], dim=0))
    ref_mha.out_proj.weight.copy_(mha.out_proj.weight)
    ref_mha.out_proj.bias.copy_(mha.out_proj.bias)
    ref_out, _ = ref_mha(x, x, x, need_weights=False)

assert torch.allclose(my_out, ref_out, atol=1e-4), "doesn't match nn.MultiheadAttention -- check the reshape/transpose steps"
print("matches nn.MultiheadAttention")

matches nn.MultiheadAttention


## Part 4: The full encoder block

TODO: build `EncoderBlock(d_model, num_heads, d_ff)` with:
- `self.attn` — a `MultiHeadAttention`
- `self.norm1` — `nn.LayerNorm(d_model)`
- `self.ff` — a feedforward network: `Linear(d_model, d_ff) -> ReLU -> Linear(d_ff, d_model)`
- `self.norm2` — another `nn.LayerNorm(d_model)`

The forward pass pattern (this is the standard "Post-LN" transformer block, and it's exactly two repetitions of the same shape): **residual add, then normalize** — twice, once around attention, once around the feedforward network. That is: compute the sublayer's output, add it back to its own input (the residual connection — this is what lets gradients flow through many stacked blocks without vanishing), *then* pass that sum through the corresponding `LayerNorm`. Both sublayers (`attn` and `ff`) follow that identical pattern; only what's inside the sublayer changes.

In [28]:
class EncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        # TODO: self.attn, self.norm1, self.ff, self.norm2
        self.attn = MultiHeadAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        # TODO: residual + norm around attn, then residual + norm around ff
        attn_out, _ = self.attn(x)
        x = self.norm1(x + attn_out)
        ff_out = self.ff(x)
        x = self.norm2(x + ff_out)
        return x

block = EncoderBlock(d_model, num_heads, d_ff)
out = block(x)
assert out.shape == (batch, seq_len, d_model), f"expected {(batch, seq_len, d_model)}, got {tuple(out.shape)}"
print("shape OK")

shape OK


## Part 5: Check against `nn.TransformerEncoderLayer`

Given, once Part 4 works. Same weight-copying idea as Part 3, extended to the feedforward and norm layers too. Note `dropout=0.0` and `.eval()` on both sides -- `nn.TransformerEncoderLayer` has dropout **on by default**, which would make this comparison fail with a real (non-bug) numerical mismatch purely from randomness, not from anything wrong in your implementation.

In [29]:
block.eval()
with torch.no_grad():
    my_block_out = block(x)

ref_block = nn.TransformerEncoderLayer(
    d_model, num_heads, dim_feedforward=d_ff, batch_first=True, norm_first=False, activation="relu", dropout=0.0
)
ref_block.eval()
with torch.no_grad():
    ref_block.self_attn.in_proj_weight.copy_(
        torch.cat([block.attn.q_proj.weight, block.attn.k_proj.weight, block.attn.v_proj.weight], dim=0)
    )
    ref_block.self_attn.in_proj_bias.copy_(
        torch.cat([block.attn.q_proj.bias, block.attn.k_proj.bias, block.attn.v_proj.bias], dim=0)
    )
    ref_block.self_attn.out_proj.weight.copy_(block.attn.out_proj.weight)
    ref_block.self_attn.out_proj.bias.copy_(block.attn.out_proj.bias)
    ref_block.linear1.weight.copy_(block.ff[0].weight); ref_block.linear1.bias.copy_(block.ff[0].bias)
    ref_block.linear2.weight.copy_(block.ff[2].weight); ref_block.linear2.bias.copy_(block.ff[2].bias)
    ref_block.norm1.weight.copy_(block.norm1.weight); ref_block.norm1.bias.copy_(block.norm1.bias)
    ref_block.norm2.weight.copy_(block.norm2.weight); ref_block.norm2.bias.copy_(block.norm2.bias)
    ref_block_out = ref_block(x)

assert torch.allclose(my_block_out, ref_block_out, atol=1e-3), "doesn't match nn.TransformerEncoderLayer"
print("matches nn.TransformerEncoderLayer -- full encoder block verified")

matches nn.TransformerEncoderLayer -- full encoder block verified


## Try yourself

1. Add `nn.Dropout(p)` in the two places `nn.TransformerEncoderLayer` uses it (after attention, after the feedforward network, both applied *before* the residual add). Verify against Part 5's reference with `dropout=0.0` still matches; then set `p>0` on both sides and confirm two calls in `.train()` mode now differ (same idea as Week 3 Day 2's `model.train()` vs `model.eval()` check).
2. Implement the **Pre-LN** variant instead: normalize *before* each sublayer rather than after (`x = x + self.attn(self.norm1(x))`, and similarly for the feedforward). Compare against `nn.TransformerEncoderLayer(..., norm_first=True)`. Modern transformers (GPT-2 onward) mostly use Pre-LN because it trains more stably in deep stacks -- can you find any discussion of why, in terms of gradient flow through the residual path?
3. Stack 3 `EncoderBlock`s in an `nn.Sequential` (or a small loop) and confirm the output shape is unchanged regardless of depth -- this is literally how a full transformer encoder is built, just many blocks deep.
4. Bring back Day 6's causal mask idea: add an optional `mask` argument to `MultiHeadAttention.forward` that gets added to the scores before `softmax`, then use it inside `EncoderBlock`. With a causal mask applied, this block starts to resemble a transformer *decoder* layer's self-attention rather than an encoder's.